# Practice: systems, rank and least squares
*AI from scratch, linear algebra 1* · lesson: https://ihor.sh/ai/l1-systems-and-inverses.html

Solving equations with NumPy, spotting rank problems in a real dataset, and fitting linear regression exactly,
three ways, on scikit-learn's diabetes data (442 patients, 10 measurements, a disease-progression score).
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Solve the linear system A @ prices = b using np.linalg.solve(A, b).'], 2: ['Compute matrix rank of Amat using np.linalg.matrix_rank(Amat).', 'Compute matrix rank of B using np.linalg.matrix_rank(B).', 'Set invertible = (r2 == 3) (a 3x3 matrix is invertible if and only if its rank is 3).'], 3: ['Solve normal equations: np.linalg.solve(X.T @ X, X.T @ y).', 'Solve with lstsq: np.linalg.lstsq(X, y, rcond=None)[0].'], 4: ['Compute rank of Xt with np.linalg.matrix_rank(Xt).', 'Compute condition number of Xt.T @ Xt with np.linalg.cond(Xt.T @ Xt).', 'Regularize G = Xt.T @ Xt + 1.0 * np.eye(Xt.shape[1]).', 'Solve G @ w_ridge = Xt.T @ y and compute cond_ridge = np.linalg.cond(G).']}
_ANSWERS = {1: 'cHJpY2VzID0gbnAubGluYWxnLnNvbHZlKEEsIGIp', 2: 'cjEgPSBucC5saW5hbGcubWF0cml4X3JhbmsoQW1hdCkKcjIgPSBucC5saW5hbGcubWF0cml4X3JhbmsoQikKaW52ZXJ0aWJsZSA9IHIyID09IDM=', 3: 'd19ub3JtYWwgPSBucC5saW5hbGcuc29sdmUoWC5UIEAgWCwgWC5UIEAgeSkKd19sc3RzcSA9IG5wLmxpbmFsZy5sc3RzcShYLCB5LCByY29uZD1Ob25lKVswXQ==', 4: 'cmFua190d2luID0gbnAubGluYWxnLm1hdHJpeF9yYW5rKFh0KQpjb25kX3R3aW4gPSBucC5saW5hbGcuY29uZChYdC5UIEAgWHQpCkcgPSBYdC5UIEAgWHQgKyAxLjAgKiBucC5leWUoWHQuc2hhcGVbMV0pCndfcmlkZ2UgPSBucC5saW5hbGcuc29sdmUoRywgWHQuVCBAIHkpCmNvbmRfcmlkZ2UgPSBucC5saW5hbGcuY29uZChHKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np

## 1. Solve a system
Three friends' café receipts: 2 coffees + 1 bagel + 1 juice = \$14; 1 coffee + 3 bagels = \$9;
2 bagels + 2 juices = \$16. The receipts are set up below as matrix `A` and bill vector `b`.
Solve the system with `np.linalg.solve` into `prices` (coffee, bagel, juice).

In [ ]:
A = np.array([[2, 1, 1], [1, 3, 0], [0, 2, 2]])
b = np.array([14, 9, 16])
prices = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print("Prices (coffee, bagel, juice):", prices)

In [ ]:
check(1, lambda: prices is not None and np.allclose(A @ prices, b), 
      "The computed prices must satisfy the receipt equations A @ prices == b.")
check(1, lambda: prices is not None and np.allclose(prices, [3, 2, 6]), 
      "Expected prices: coffee = $3, bagel = $2, juice = $6.")

## 2. Rank: which equations count?
`np.linalg.matrix_rank` counts independent rows (= independent columns). Set `r1` and `r2` to the ranks of the
two matrices below using `matrix_rank`. Then: can `B` be inverted? Set `invertible` to `True` or `False`.

In [ ]:
Amat = np.array([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
B = np.array([[1, 2, 3], [2, 4, 6], [1, 0, 1]])
r1 = r2 = invertible = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print("Rank Amat:", r1, "Rank B:", r2, "B invertible:", invertible)

In [ ]:
check(2, lambda: (r1, r2) == (3, 2), 
      "Amat has 3 linearly independent rows (rank 3); B has row 2 = 2 * row 1 (rank 2).")
check(2, lambda: invertible is False or invertible == False, 
      "Matrix B has rank 2 < 3, so its determinant is 0 and it cannot be inverted (invertible = False).")

## 3. Least squares
Load the data. `X` is 442 × 10; add a column of 1s for the intercept.

In [ ]:
from sklearn.datasets import load_diabetes
data = load_diabetes()
X = np.column_stack([data.data, np.ones(len(data.data))])
y = data.target
print(X.shape, y.shape, data.feature_names)

Fit $w$ using linear algebra formulas:
- `w_normal`: solve the normal equations $X^\top X w = X^\top y$ using `solve` (`np.linalg.solve`)
- `w_lstsq`: `lstsq` (`np.linalg.lstsq(X, y, rcond=None)[0]`, more careful about rounding: uses SVD)

In [ ]:
from sklearn.linear_model import LinearRegression
w_normal = w_lstsq = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
lr = LinearRegression().fit(data.data, y)
w_sklearn = np.append(lr.coef_, lr.intercept_)
print("Normal equations weights:\n", w_normal)

In [ ]:
check(3, lambda: w_normal is not None and w_lstsq is not None and np.allclose(w_normal, w_lstsq) and np.allclose(w_lstsq, w_sklearn), 
      "All three approaches (normal equations, lstsq, sklearn LinearRegression) should produce identical weights.")
check(3, lambda: w_lstsq is not None and np.isclose(np.mean((X @ w_lstsq - y) ** 2), 2859.7, atol=1), 
      "The mean squared error on training features should be approximately 2860.")

## 4. A twin column breaks it
Add a column that's an exact mix of two others (`bmi + bp`). Compute `rank_twin` with `matrix_rank` and `cond_twin`, the
**condition number** of $X^\top X$ using `cond` (`np.linalg.cond`): how much rounding errors get magnified. A huge number
means "nearly singular": the weights stop meaning anything.
Then fix it the regularization way: add $\lambda I$ to $X^\top X$ before solving using `eye` (`np.eye(Xt.shape[1])`).
Set `w_ridge` with $\lambda = 1$ and `cond_ridge` for the fixed matrix.

In [ ]:
Xt = np.column_stack([X, X[:, 2] + X[:, 3]])
rank_twin = cond_twin = w_ridge = cond_ridge = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("rank", rank_twin, "of", Xt.shape[1], " cond", cond_twin, "→", cond_ridge)

In [ ]:
check(4, lambda: rank_twin == 11, 
      "Xt has 12 columns but only 11 independent dimensions (since column 12 is collinear with 2 + 3).")
check(4, lambda: cond_twin is not None and cond_twin > 1e12, 
      "Collinear features cause Xt.T @ Xt to become nearly singular with condition number > 1e12.")
check(4, lambda: cond_ridge is not None and cond_ridge < 1e4 and np.isfinite(w_ridge).all(), 
      "Adding lambda * I (Ridge regularization) restores stability and drops the condition number below 1e4.")

## Next
Back to the path: https://ihor.sh/ai/ · next: chapter 2, gradients.